# 05 — Bộ đánh giá, Popularity và Repeat-last

Bước G của pipeline: dựng **một** đường đánh giá dùng chung cho mọi mô hình và chạy hai baseline.

- Metric (`rr.metrics`): Hit, Recall, MRR, NDCG tại `K = 20`, target là tập item; chỉ số chính **NDCG@20**. Mẫu số là mọi query, kể cả query có target ngoài vocabulary (0 điểm).
- Ứng viên tại `t` (`rr.candidates`): có trong vocabulary train và `known_at_train_ms <= t`. Áp dụng cho cả mô hình lẫn fallback.
- Bài toán: tương tác kế tiếp — target là tập item phân biệt của nhóm `t+1`, mọi hành vi.
- Popularity: (a) đếm đều; (b) trọng số hành vi `w_view = 1`, lưới `w_cart, w_buy ∈ {1, 2, 5, 10}`, chọn trên **validation**.
- Repeat-last (D1): item trong prefix theo nhóm gần nhất, hòa thì hành vi mạnh hơn, rồi itemid; phần còn thiếu lấp bằng fallback.
- Đầu ra: `reports/05_08_validation/popularity_validation.json`, `reports/05_08_validation/repeat_last_validation.json`.

Chỉ dùng train để fit và validation để chọn. Notebook này **không đọc test**.

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time, itertools

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.metrics import per_query_metrics, validate_topk, metric_columns
from rr.candidates import CandidateFilter
from rr.samples import prefix_events
from rr.models import popularity, repeat_last
from rr.evaluation import load_samples, run, flat

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/05_08_validation"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
K = max(PROTOCOL["evaluation"]["ks"])

samples_report = read_json(ROOT / "reports/04_samples/metrics.json")
metadata_report = read_json(ROOT / "reports/03_metadata/metrics.json")
split_report = read_json(ROOT / "reports/02_session_split/metrics.json")
require(samples_report["protocol"]["version"] == PROTOCOL["version"],
        "reports/04_samples được tạo với protocol khác; chạy lại notebook 04.")
expected = {p: samples_report["artifacts"][p] for p in [
    "data/processed/item_vocab.parquet", "data/processed/samples/train_queries.parquet",
    "data/processed/samples/validation_queries.parquet", "data/processed/samples/validation_targets.parquet"]}
expected["data/processed/item_property_history.parquet"] = metadata_report["artifacts"]["data/processed/item_property_history.parquet"]
expected["data/processed/item_catalog.parquet"] = metadata_report["artifacts"]["data/processed/item_catalog.parquet"]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| protocol", PROTOCOL["version"])

## 2. Ví dụ tính tay cho metric (6 ca bắt buộc)

Một query có `G = {7}` hoặc `G = {7, 8}`, K = 3. Với `log2(2) = 1`, `log2(3) ≈ 1,585`, `log2(4) = 2`:

| Ca | Top-3 | G | hit | recall | MRR | NDCG |
| --- | --- | --- | --- | --- | --- | --- |
| không hit | 1 2 3 | {7} | 0 | 0 | 0 | 0 |
| hit rank 1 | 7 2 3 | {7} | 1 | 1 | 1 | 1 |
| hit rank K | 1 2 7 | {7} | 1 | 1 | 1/3 | 1/2 |
| nhiều target | 7 1 8 | {7, 8} | 1 | 1 | 1 | (1 + 1/2)/(1 + 1/1,585) |
| K > số ứng viên | 8 -1 -1 | {7, 8} | 1 | 1/2 | 1 | 1/(1 + 1/1,585) |
| trùng item | 7 7 3 | — | bị từ chối (ValueError) | | | |

In [ ]:
_topk = np.array([[1, 2, 3], [7, 2, 3], [1, 2, 7], [7, 1, 8], [8, -1, -1]])
_targets = pd.DataFrame({"query_id": [0, 1, 2, 3, 3, 4, 4], "itemid": [7, 7, 7, 7, 8, 7, 8]})
_m = per_query_metrics(_topk, np.arange(5), _targets, ks=(3,))
_idcg2 = 1 + 1 / np.log2(3)
np.testing.assert_allclose(_m["hit@3"], [0, 1, 1, 1, 1])
np.testing.assert_allclose(_m["recall@3"], [0, 1, 1, 1, 0.5])
np.testing.assert_allclose(_m["mrr@3"], [0, 1, 1 / 3, 1, 1])
np.testing.assert_allclose(_m["ndcg@3"], [0, 1, 0.5, 1.5 / _idcg2, 1 / _idcg2])
try:
    validate_topk(np.array([[7, 7, 3]]))
    raise AssertionError("Phải từ chối item trùng.")
except ValueError:
    pass
# query có target ngoài vocabulary vẫn nằm trong mẫu số (0 điểm)
_m2 = per_query_metrics(np.array([[1, 2, 3], [7, 2, 3]]), [0, 1], pd.DataFrame({"query_id": [0, 1], "itemid": [999, 7]}), ks=(3,))
assert _m2["ndcg@3"].mean() == 0.5
del _topk, _targets, _m, _m2
print("Đã kiểm tra 6 ca tính tay và mẫu số có target ngoài vocabulary.")

## 3. Dữ liệu fit (train) và bộ lọc ứng viên

`fallback_top_n` item phổ biến nhất được quét cho mỗi query (giới hạn bộ nhớ). Sau mỗi lần chạy, notebook kiểm tra mọi query vẫn đủ K item.

In [ ]:
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
train_events = events[events["split"] == "train"].reset_index(drop=True)
require(set(train_events["itemid"].unique()) == set(vocab["itemid"]), "Vocabulary không khớp item của train.")
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
val_q, val_t = load_samples(ROOT, "validation")
require(val_q["query_time_ms"].min() > train_events["timestamp"].max(), "Validation phải nằm sau toàn bộ train.")
print(f"Train: {len(train_events):,} sự kiện | validation: {len(val_q):,} query, {len(val_t):,} lượt target")
print(f"Target validation đủ điều kiện ứng viên: {val_t['candidate_eligible'].mean():.2%}")

## 4. Popularity: đếm đều và lưới trọng số hành vi

In [ ]:
EMPTY = np.empty((len(val_q), 0), dtype=np.int64)
grid = PROTOCOL["behavior_weights"]["grid"]
configs = [dict(view=v, addtocart=c, transaction=b)
           for v, c, b in itertools.product(grid["view"], grid["addtocart"], grid["transaction"])]
pop_reports, pop_rankings, timings = {}, {}, {}
for w in configs:
    name = f"pop_v{w['view']}_c{w['addtocart']}_b{w['transaction']}"
    t0 = time.perf_counter()
    items, scores = popularity(train_events, w)
    fit_s = time.perf_counter() - t0
    report, per_q, topk, _ = run(EMPTY, None, val_q, val_t, flt, items[:FALLBACK_TOP_N], scores[:FALLBACK_TOP_N])
    require(report["short_lists"] == 0, f"{name}: có query thiếu item sau lọc; tăng fallback_top_n.")
    report["weights"], report["fit_seconds"] = w, round(fit_s, 2)
    pop_reports[name], pop_rankings[name] = report, (items, scores)
table = flat(pop_reports).sort_values("ndcg@20", ascending=False)
display(table.style.format("{:.4f}"))

**Chọn cấu hình.** Cấu hình có NDCG@20 validation cao nhất được dùng làm Popularity "theo hành vi". Theo protocol, fallback dùng trọng số đều, trừ khi trọng số đã tune tốt hơn trên validation.

In [ ]:
UNIFORM = "pop_v1_c1_b1"
best_pop = table.index[0]
fallback_name = best_pop if pop_reports[best_pop]["all"]["ndcg@20"] > pop_reports[UNIFORM]["all"]["ndcg@20"] else UNIFORM
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in pop_rankings[fallback_name])
print(f"Popularity tốt nhất: {best_pop} | đều: {UNIFORM} | fallback dùng: {fallback_name}")
display(flat({k: pop_reports[k] for k in dict.fromkeys([UNIFORM, best_pop])}, "all").style.format("{:.4f}"))

## 5. Repeat-last (D1)

Repeat-last đo mức điểm có được chỉ nhờ gợi ý lại item đã xem trong tiền tố phiên (tách Target mới vs Target lặp).

In [ ]:
prefix = prefix_events(val_q, events[events["split"] == "validation"], columns=("itemid", "event"))
t0 = time.perf_counter()
rl_items, rl_scores = repeat_last(prefix, val_q["query_id"].to_numpy(), width=50)
rl_time = time.perf_counter() - t0
rl_report, rl_per_q, rl_topk, _ = run(rl_items, rl_scores, val_q, val_t, flt, fb_items, fb_scores)
rl_report["query_seconds_total"] = round(rl_time, 2)
require(rl_report["short_lists"] == 0, "Repeat-last có query thiếu item.")

summary = {"popularity_uniform": pop_reports[UNIFORM], f"popularity_best ({best_pop})": pop_reports[best_pop],
           "repeat_last": rl_report}
for part in ["all", "new_targets", "repeat_targets", "strong_behavior_targets", "view_only_targets"]:
    print(part)
    display(flat(summary, part)[["hit@20", "recall@20", "mrr@20", "ndcg@20"]].style.format("{:.4f}"))

## 6. Theo độ dài prefix (D3)

In [ ]:
bins = {name: r["by_prefix_len"] for name, r in summary.items()}
display(pd.DataFrame({name: {b: v["ndcg@20"] for b, v in r.items()} for name, r in bins.items()}).style.format("{:.4f}"))
display(pd.Series({b: v["n_queries"] for b, v in rl_report["by_prefix_len"].items()}, name="n_queries"))

## 7. Lưu báo cáo

Cấu hình tốt nhất của Popularity và quy tắc fallback được ghi lại để các notebook sau dùng đúng cùng một fallback.

In [ ]:
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
common = {"generated_at_utc": utc_now(), **environment(), "notebook": "notebooks/05_danh_gia_va_popularity.ipynb",
          "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
          "inputs": inputs, "split": "validation", "fit_data": "train", "n_queries": len(val_q),
          "fallback_top_n": FALLBACK_TOP_N}
write_json(EVAL_DIR / "popularity_validation.json", {
    **common, "grid": pop_reports, "uniform": UNIFORM, "best": best_pop,
    "selected": {"popularity_behavior_weights": pop_reports[best_pop]["weights"], "fallback": fallback_name,
                 "fallback_weights": pop_reports[fallback_name]["weights"]},
    "status": "PASS"})
write_json(EVAL_DIR / "repeat_last_validation.json", {
    **common, "fallback": fallback_name, "report": rl_report, "status": "PASS"})
print("Báo cáo:", rel(EVAL_DIR / "popularity_validation.json", ROOT), rel(EVAL_DIR / "repeat_last_validation.json", ROOT))